# Museum Analysis

This notebook connects to the museum database and conducts simple analysis of the data.

## Imports

In [7]:
import pandas as pd
import altair as alt
from psycopg2 import connect
from dotenv import dotenv_values
from psycopg2.extras import RealDictCursor

config = dotenv_values(".env")

## Setup

### DB Connection

In [8]:
conn = connect(
    host=config["DB_HOST"],
    port=config["DB_PORT"],
    dbname=config["DB_NAME"],
    user=config["DB_USER"],
    password=config["DB_PASSWORD"],
    cursor_factory=RealDictCursor
)

## Prenotes

- Data collected from the exhibition 'Cetacean Sensations' did not appear in the S3 bucket. This may have affected the accuracy of the visualisations for requests/ratings.

- Data has been cleaned to remove dates before 8:45 AM and after 6:15 PM to account for any accidental presses of kiosk's buttons.
- Current available data is only available for the 6th of March 2023.

## Visualisations - Assistance Requests by Exhibition

The number of requests for each exhibition is visualised in the bar chart below.

We can see that 'The Crenshaw Collection' has the most assistance requests by over double second place 'Thunder Lizards'.

In [9]:
query = """
    SELECT
    e.exhibition_name,
        COUNT(r.exhibition_id) AS request_count
    FROM exhibitions e
    LEFT JOIN requests r 
        ON e.exhibition_id = r.exhibition_id 
    AND r.request_value = 0
    GROUP BY e.exhibition_id, e.exhibition_name
    ORDER BY request_count DESC, e.exhibition_name ASC;
    """
with conn.cursor() as cursor:
    cursor.execute(query)
    rows = cursor.fetchall()
    colnames = [desc[0] for desc in cursor.description]
    df = pd.DataFrame(rows, columns=colnames)

chart = alt.Chart(df).mark_bar().encode(
    x=alt.X('exhibition_name:N', title='Exhibition Name', sort='-y'),
    y=alt.Y('request_count:Q', title='Request Count'),
    color=alt.Color('request_count:Q', scale=alt.Scale(
        scheme='reds'), legend=None)
).properties(
    title='Assistance Request Count by Exhibition',
    width=400
)

chart

alt.Chart(...)

## Visualisations - Emergency Requests by Exhibition

The number of emergency requests for each exhibition is visualised in the bar chart below.

We can see that 'Thunder Lizards' has the most emergency requests. In fact it has the only emergency requests among all exhibitions.

In [10]:
query = """
    SELECT
    e.exhibition_name,
        COUNT(r.exhibition_id) AS request_count
    FROM exhibitions e
    LEFT JOIN requests r 
        ON e.exhibition_id = r.exhibition_id 
    AND r.request_value = 1
    GROUP BY e.exhibition_id, e.exhibition_name
    ORDER BY request_count DESC, e.exhibition_name ASC;
    """
with conn.cursor() as cursor:
    cursor.execute(query)
    rows = cursor.fetchall()
    colnames = [desc[0] for desc in cursor.description]
    df = pd.DataFrame(rows, columns=colnames)

chart = alt.Chart(df).mark_bar().encode(
    x=alt.X('exhibition_name:N', title='Exhibition Name', sort='-y'),
    y=alt.Y('request_count:Q', title='Request Count'),
    color=alt.Color('request_count:Q', scale=alt.Scale(
        scheme='reds'), legend=None)
).properties(
    title='Emergency Request Count by Exhibition',
    width=400
)

chart

alt.Chart(...)

## Visualisation of Number of Ratings by Exhibition

We can see that the number of ratings varies across different exhibitions, with some exhibitions receiving significantly more ratings than others. I will further investigate the factors contributing to this variation in the subsequent analysis.

However possible reasons for this variation could include differences in exhibition popularity, exhibition start date, or the visibility of the rating kiosks within each exhibition.

In [11]:
query = """
    SELECT
    e.exhibition_name,
        COUNT(r.exhibition_id) AS request_count
    FROM exhibitions e
    LEFT JOIN ratings r
        ON e.exhibition_id = r.exhibition_id 
    AND r.rating_value = 1
    GROUP BY e.exhibition_id, e.exhibition_name
    ORDER BY request_count DESC, e.exhibition_name ASC;
    """
with conn.cursor() as cursor:
    cursor.execute(query)
    rows = cursor.fetchall()
    colnames = [desc[0] for desc in cursor.description]
    df = pd.DataFrame(rows, columns=colnames)

chart = alt.Chart(df).mark_bar().encode(
    x=alt.X('exhibition_name:N', title='Exhibition Name', sort='-y'),
    y=alt.Y('request_count:Q', title='Rating Count'),
    color=alt.Color('request_count:Q', scale=alt.Scale(
        scheme='reds'), legend=None)
).properties(
    title='Rating Count by Exhibition',
    width=400
)

chart

alt.Chart(...)

## Visualisations - Average Rating by Exhibition Over Time

We can see how the average rating for each exhibition changes over the course of the day. The line chart allows us to identify trends and patterns in visitor ratings at different hours.

We can see that 'The Crenshaw Collection' tends to have higher average ratings during the morning hours, while other exhibitions show more consistent ratings throughout the day.

'Cetacean Sensations' is purposefully not included in the graph as there are no ratings available for this exhibition during the observed time period.


In [26]:
query = """
    SELECT
        EXTRACT(HOUR FROM r.time_at) AS rating_hour,
        e.exhibition_name,
        AVG(r.rating_value)::FLOAT AS avg_rating
    FROM exhibitions e
    JOIN ratings r USING (exhibition_id)
    GROUP BY EXTRACT(HOUR FROM r.time_at), e.exhibition_name
    ORDER BY rating_hour ASC;
    """
with conn.cursor() as cursor:
    cursor.execute(query)
    rows = cursor.fetchall()
    colnames = [desc[0] for desc in cursor.description]
    df = pd.DataFrame(rows, columns=colnames)

df

df['rating_hour'] = df['rating_hour'].astype(int)
df['rating_time'] = pd.to_datetime(df['rating_hour'].astype(int), format='%H')
df['avg_rating'] = df['avg_rating'].astype(float)

chart = alt.Chart(df).mark_line(point=True).encode(
    x=alt.X(
        'rating_time:T',
        title='Rating Hour',
        # Formats as '08 AM', '01 PM', etc.
        axis=alt.Axis(format='%I %p', labelAngle=0)
    ),
    y=alt.Y('avg_rating:Q', title='Average Rating',
            scale=alt.Scale(domain=[0, 4])),
    color=alt.Color('exhibition_name:N', title='Exhibition Name'),
    tooltip=[
        'exhibition_name',
        alt.Tooltip('rating_time:T', format='%I:%M %p', title='Hour'),
        alt.Tooltip('avg_rating:Q', format='.2f', title='Avg Rating')
    ]
).properties(
    title='Average Rating by Exhibition Over Time',
    width=650,
    height=400
)

chart

alt.Chart(...)